In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data, util

def load_image(path=''):
    if not path.strip():
        return data.camera()
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4), squeeze=False)
    for ax, im, title in zip(axes.ravel(), images, titles):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

img = load_image(input('Image path (Enter for sample): '))

In [ ]:
# 1
hist = np.bincount(img.ravel(), minlength=256)
cdf = hist.cumsum()
nonzero = cdf[hist > 0]
lo = nonzero[0]
lut = np.round(np.clip((cdf-lo)/max(cdf[-1]-lo,1),0,1)*255).astype(np.uint8)
equalized = lut[img] if cdf[-1] != lo else img.copy()
show([img,equalized],['Original','Equalized'])

In [ ]:
# 2
path = input('Reference image path (Enter for sample): ')
ref = load_image(path) if path.strip() else data.moon()
def cdf_of(im):
    h = np.bincount(im.ravel(), minlength=256)
    return h.cumsum()/im.size
source_cdf, ref_cdf = cdf_of(img), cdf_of(ref)
lut = np.argmin(np.abs(source_cdf[:,None]-ref_cdf[None,:]),axis=1).astype(np.uint8)
matched = lut[img]
show([img,ref,matched],['Original','Reference','Matched'])